In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
# N-BEATSx-like with exogenous covariates (PyTorch, 개선버전)
# - Train: ./dataset/train.csv
# - Test : ./dataset/TEST_*.csv (각 파일: 28일 입력 -> 7일 예측)
# - Out  : ./result/nbeatsx_exog_forecasts_long.csv
# 학습 손실: MAE(원 스케일), 검증: sMAPE, EarlyStopping(sMAPE)
# 외생변수: DOW(one-hot+sin/cos), weekend, month(sin/cos), quarter(one-hot),
#           holiday(KR), prev/next-holiday, EOM, payday(25일), DOM(sin/cos)
# 주의: 대회 규정 상 외부 데이터 불가. 공휴일은 도메인 지식으로 허용.

# ========== 설치 및 import ==========
try:
    from tqdm.auto import tqdm
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    from tqdm.auto import tqdm

import os, glob, math, random, sys, subprocess
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    import holidays
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "holidays"])
    import holidays

# ========== 설정 ==========
DATASET_DIR = "./dataset"
RESULT_DIR = "./result"
os.makedirs(RESULT_DIR, exist_ok=True)

BACKCAST_LEN = 28
FORECAST_LEN = 7

HIDDEN_UNITS = 1024
NB_BLOCKS = 4
NB_STACKS = 3

EPOCHS = 50
PATIENCE = 8                 # sMAPE early-stopping
ES_MIN_DELTA = 1e-5          # 개선 최소폭
BATCH_SIZE = 4096
LR = 2e-3
WEIGHT_DECAY = 1e-6
VAL_RATIO = 0.1
SEED = 2025
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPS = 1e-6
HOLIDAY_COUNTRY = "KR"
PAYDAY_DOM = 25

torch.set_float32_matmul_precision("high")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ========== 캘린더 피처 ==========
def _build_calendar_features(dates) -> np.ndarray:
    dates = pd.DatetimeIndex(dates)

    dow = np.asarray(dates.weekday)                   # 0..6
    dow_oh = np.eye(7, dtype=np.float32)[dow]         # (N,7)
    ang = 2*np.pi
    dow_sin = np.sin(ang*dow/7.0)[:,None].astype(np.float32)
    dow_cos = np.cos(ang*dow/7.0)[:,None].astype(np.float32)
    weekend = (dow >= 5).astype(np.float32)[:,None]

    month = np.asarray(dates.month)                   # 1..12
    m_sin = np.sin(ang*(month-1)/12.0)[:,None].astype(np.float32)
    m_cos = np.cos(ang*(month-1)/12.0)[:,None].astype(np.float32)
    quarter = ((month-1)//3).astype(int)              # 0..3
    q_oh = np.eye(4, dtype=np.float32)[quarter]       # (N,4)

    dom = np.asarray(dates.day)                       # 1..31
    dom_sin = np.sin(ang*(dom-1)/31.0)[:,None].astype(np.float32)
    dom_cos = np.cos(ang*(dom-1)/31.0)[:,None].astype(np.float32)

    eom = dates.is_month_end.astype(np.float32)[:,None]

    # 공휴일(도메인 지식)
    try:
        hol_set = holidays.country_holidays(HOLIDAY_COUNTRY)
        is_hol   = np.array([(d.date() in hol_set) for d in dates], dtype=np.float32)[:,None]
        prev_hol = np.array([((d - pd.Timedelta(days=1)).date() in hol_set) for d in dates], dtype=np.float32)[:,None]
        next_hol = np.array([((d + pd.Timedelta(days=1)).date() in hol_set) for d in dates], dtype=np.float32)[:,None]
    except Exception:
        is_hol = np.zeros((len(dates),1), np.float32)
        prev_hol = np.zeros((len(dates),1), np.float32)
        next_hol = np.zeros((len(dates),1), np.float32)

    payday = (dom == PAYDAY_DOM).astype(np.float32)[:,None]

    feats = np.concatenate([
        dow_oh, dow_sin, dow_cos,
        weekend,
        m_sin, m_cos,
        q_oh,
        is_hol, prev_hol, next_hol,
        eom, payday,
        dom_sin, dom_cos
    ], axis=1).astype(np.float32)
    return feats

# ========== 데이터 로드 ==========
train_path = os.path.join(DATASET_DIR, "train.csv")
assert os.path.exists(train_path), "dataset/train.csv 이 없습니다."
train_df = pd.read_csv(train_path)
need = {"date","store","menu","store_menu","sales"}
miss = need - set(train_df.columns)
if miss: raise ValueError(f"train.csv 누락 컬럼: {sorted(miss)}")

train_df["date"] = pd.to_datetime(train_df["date"])
train_df = train_df.sort_values(["store_menu","date"]).reset_index(drop=True)
train_df["sales"] = train_df["sales"].astype(float).clip(lower=0)

# 스케일러(시계열별 표준화)
scalers = {}
for sm, g in train_df.groupby("store_menu", sort=False):
    v = g["sales"].values.astype(np.float32)
    mu = float(v.mean()); sd = float(v.std()); sd = 1.0 if sd==0 else sd
    scalers[sm] = (mu, sd)
global_mu = float(train_df["sales"].mean())
global_sd = float(train_df["sales"].std()) if train_df["sales"].std()>0 else 1.0
def get_scale(sm): return scalers.get(sm, (global_mu, global_sd))

# 훈련 구간 날짜 피처 캐시
uniq_dates = pd.DatetimeIndex(sorted(train_df["date"].unique()))
cal_feat = _build_calendar_features(uniq_dates)
feat_map = {d: cal_feat[i] for i, d in enumerate(uniq_dates)}
E_DIM = cal_feat.shape[1]

# 윈도우 생성
def make_windows_with_exog(df, backcast_len=BACKCAST_LEN, forecast_len=FORECAST_LEN):
    Xy, Xb_ex, Xf_ex, Yf, MU, SD = [], [], [], [], [], []
    for sm, g in df.groupby("store_menu", sort=False):
        vals = g["sales"].values.astype(np.float32)
        dates = g["date"].values
        if len(vals) < backcast_len + forecast_len: 
            continue
        mu, sd = get_scale(sm)
        vals_std = (vals - mu) / sd
        L = len(vals) - (backcast_len + forecast_len) + 1
        for i in range(L):
            b = slice(i, i+backcast_len)
            h = slice(i+backcast_len, i+backcast_len+forecast_len)
            Xy.append(vals_std[b])
            Yf.append(vals_std[h])
            bd = pd.DatetimeIndex(dates[b])
            hd = pd.DatetimeIndex(dates[h])
            Xb_ex.append(np.stack([feat_map[d] for d in bd], axis=0))
            Xf_ex.append(np.stack([feat_map[d] for d in hd], axis=0))
            MU.append(mu); SD.append(sd)
    if not Xy:
        return (np.empty((0,backcast_len),np.float32),
                np.empty((0,backcast_len,E_DIM),np.float32),
                np.empty((0,forecast_len,E_DIM),np.float32),
                np.empty((0,forecast_len),np.float32),
                np.empty((0,),np.float32),
                np.empty((0,),np.float32))
    return (np.stack(Xy), np.stack(Xb_ex), np.stack(Xf_ex),
            np.stack(Yf), np.array(MU,np.float32), np.array(SD,np.float32))

X_y, X_bex, X_fex, Y_f, MU_all, SD_all = make_windows_with_exog(train_df)
assert len(X_y)>0, "학습 윈도우가 비어있습니다."

# Train/Val split
n = len(X_y)
perm = np.random.permutation(n)
val_n = int(n * VAL_RATIO)
val_idx, tr_idx = perm[:val_n], perm[val_n:]
def _pick(a, idc): return a[idc] if len(a)>0 else a
Xy_tr, Xb_tr, Xf_tr, Yf_tr, MU_tr, SD_tr = _pick(X_y,tr_idx), _pick(X_bex,tr_idx), _pick(X_fex,tr_idx), _pick(Y_f,tr_idx), MU_all[tr_idx], SD_all[tr_idx]
Xy_va, Xb_va, Xf_va, Yf_va, MU_va, SD_va = _pick(X_y,val_idx), _pick(X_bex,val_idx), _pick(X_fex,val_idx), _pick(Y_f,val_idx), MU_all[val_idx], SD_all[val_idx]

class WinDS(Dataset):
    def __init__(self, Xy, Xb, Xf, Yf, MU, SD):
        self.Xy  = torch.from_numpy(Xy)      # (N,B)
        self.Xb  = torch.from_numpy(Xb)      # (N,B,E)
        self.Xf  = torch.from_numpy(Xf)      # (N,H,E)
        self.Yf  = torch.from_numpy(Yf)      # (N,H)
        self.MU  = torch.from_numpy(MU)      # (N,)
        self.SD  = torch.from_numpy(SD)      # (N,)
    def __len__(self): return len(self.Xy)
    def __getitem__(self, i):
        return self.Xy[i], self.Xb[i], self.Xf[i], self.Yf[i], self.MU[i], self.SD[i]

num_workers = max(1, os.cpu_count()//2)
pin = (DEVICE=="cuda")
tr_loader  = DataLoader(WinDS(Xy_tr, Xb_tr, Xf_tr, Yf_tr, MU_tr, SD_tr), batch_size=BATCH_SIZE, shuffle=True,  drop_last=False, num_workers=num_workers, pin_memory=pin)
val_loader = DataLoader(WinDS(Xy_va, Xb_va, Xf_va, Yf_va, MU_va, SD_va), batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=num_workers, pin_memory=pin)

# ========== N-BEATSx-like 모델(완전 벡터화) ==========
class NBeatsXBlock(nn.Module):
    def __init__(self, backcast_len, forecast_len, e_dim, hidden_units=1024):
        super().__init__()
        B, H, E = backcast_len, forecast_len, e_dim
        in_dim = B + B*E + H*E
        self.fc1 = nn.Linear(in_dim, hidden_units)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.fc3 = nn.Linear(hidden_units, hidden_units)
        self.fc4 = nn.Linear(hidden_units, hidden_units)
        self.backcast_head = nn.Linear(hidden_units, B)
        self.forecast_head = nn.Linear(hidden_units, H)
        self.act = nn.ReLU(inplace=True)
        for m in [self.fc1,self.fc2,self.fc3,self.fc4,self.backcast_head,self.forecast_head]:
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, y_b, xb_exog, xf_exog):
        # y_b: (N,B), xb_exog:(N,B,E), xf_exog:(N,H,E)
        N = y_b.size(0)
        z = torch.cat([y_b, xb_exog.flatten(1), xf_exog.flatten(1)], dim=1)  # (N, in_dim)
        h = self.act(self.fc1(z))
        h = self.act(self.fc2(h))
        h = self.act(self.fc3(h))
        h = self.act(self.fc4(h))
        back = self.backcast_head(h)   # (N,B)
        fore = self.forecast_head(h)   # (N,H)
        return back, fore

class NBeatsX(nn.Module):
    def __init__(self, backcast_len, forecast_len, e_dim, nb_stacks=3, nb_blocks=4, hidden_units=1024):
        super().__init__()
        self.B = backcast_len; self.H = forecast_len
        blocks = []
        for _ in range(nb_stacks):
            for _ in range(nb_blocks):
                blocks.append(NBeatsXBlock(backcast_len, forecast_len, e_dim, hidden_units))
        self.blocks = nn.ModuleList(blocks)

    def forward(self, y_b, xb_exog, xf_exog):
        residual = y_b
        fsum = torch.zeros((y_b.size(0), self.H), device=y_b.device, dtype=y_b.dtype)
        for blk in self.blocks:
            back, fore = blk(residual, xb_exog, xf_exog)   # 벡터화
            residual = residual - back
            fsum = fsum + fore
        return fsum  # (N,H), standardized

model = NBeatsX(BACKCAST_LEN, FORECAST_LEN, E_DIM, NB_STACKS, NB_BLOCKS, HIDDEN_UNITS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-5)

use_amp = (DEVICE=="cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
MAX_GRAD_NORM = 1.0

# ========== 손실/지표 ==========
def mae_from_std(yhat_std, y_std, mu, sd):
    yhat = yhat_std * sd.unsqueeze(1) + mu.unsqueeze(1)
    y    = y_std    * sd.unsqueeze(1) + mu.unsqueeze(1)
    return torch.mean(torch.abs(yhat - y))

def smape_from_std(yhat_std, y_std, mu, sd, eps=EPS):
    yhat = yhat_std * sd.unsqueeze(1) + mu.unsqueeze(1)
    y    = y_std    * sd.unsqueeze(1) + mu.unsqueeze(1)
    num = 2.0 * torch.abs(yhat - y)
    den = torch.abs(yhat) + torch.abs(y) + eps
    return torch.mean(num / den)

# ========== 학습 루프 ==========
best_val = float("inf")
best_state = None
bad = 0

for epoch in range(1, EPOCHS+1):
    # train
    model.train()
    tr_loss_sum, n_tr = 0.0, 0
    pbar = tqdm(tr_loader, desc=f"[{epoch:02d}] train", leave=False)
    for xb_y, xb_ex, xf_ex, y_f, mu, sd in pbar:
        xb_y, xb_ex, xf_ex, y_f = xb_y.to(DEVICE), xb_ex.to(DEVICE), xf_ex.to(DEVICE), y_f.to(DEVICE)
        mu, sd = mu.to(DEVICE), sd.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            yhat = model(xb_y, xb_ex, xf_ex)
            loss = mae_from_std(yhat, y_f, mu, sd)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer); scaler.update()

        bs = xb_y.size(0)
        tr_loss_sum += loss.item()*bs; n_tr += bs
        pbar.set_postfix(mae=f"{(tr_loss_sum/n_tr):.4f}")

    tr_loss = tr_loss_sum / max(1,n_tr)

    # val
    model.eval()
    val_sum, n_va = 0.0, 0
    with torch.no_grad():
        for xb_y, xb_ex, xf_ex, y_f, mu, sd in tqdm(val_loader, desc=f"[{epoch:02d}] val", leave=False):
            xb_y, xb_ex, xf_ex, y_f = xb_y.to(DEVICE), xb_ex.to(DEVICE), xf_ex.to(DEVICE), y_f.to(DEVICE)
            mu, sd = mu.to(DEVICE), sd.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16 if use_amp else None):
                yhat = model(xb_y, xb_ex, xf_ex)
                smape_val = smape_from_std(yhat, y_f, mu, sd).item()  # float
            bs = int(xb_y.size(0))
            val_sum += float(smape_val) * bs
            n_va += bs

    val_smape = val_sum / max(1, n_va)
    scheduler.step(val_smape)

    improved = (best_val - val_smape) > ES_MIN_DELTA
    if improved:
        best_val = val_smape
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    print(f"[{epoch:02d}] train MAE {tr_loss:.6f} | val sMAPE {val_smape:.6f} | best {best_val:.6f} | patience {bad}/{PATIENCE}")
    if bad >= PATIENCE:
        print("Early stopping.")
        break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
model.eval()

# ========== 추론 ==========
from functools import lru_cache

@lru_cache(maxsize=4096)
def _feats_for_dates_cached(start_date_str, periods):
    start = pd.to_datetime(start_date_str)
    dates = pd.date_range(start, periods=periods, freq="D")
    return _build_calendar_features(dates)

def _build_feats_for_dates(dates: pd.DatetimeIndex) -> np.ndarray:
    # 캐시 키는 시작일+길이
    return _feats_for_dates_cached(dates.min().date().isoformat(), len(dates))

def forecast_with_exog(sm: str, last28_dates: pd.DatetimeIndex, last28_vals: np.ndarray):
    mu, sd = get_scale(sm)
    y_std = (last28_vals - mu) / sd
    fut_dates = pd.date_range(last28_dates.max() + pd.Timedelta(days=1), periods=FORECAST_LEN, freq="D")

    xb_exog = _build_feats_for_dates(last28_dates)  # (B,E)
    xf_exog = _build_feats_for_dates(fut_dates)     # (H,E)

    xb_y  = torch.from_numpy(y_std.astype(np.float32)[None,:]).to(DEVICE)
    xb_ex = torch.from_numpy(xb_exog.astype(np.float32)[None,:,:]).to(DEVICE)
    xf_ex = torch.from_numpy(xf_exog.astype(np.float32)[None,:,:]).to(DEVICE)

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16 if use_amp else None):
        yhat_std = model(xb_y, xb_ex, xf_ex).float().cpu().numpy()[0]
    yhat = yhat_std * sd + mu
    return fut_dates, np.clip(yhat, 0.0, None)

test_files = sorted(glob.glob(os.path.join(DATASET_DIR, "TEST_*.csv")))
if not test_files:
    print("경고: dataset/TEST_*.csv 이 없습니다. 예측은 건너뜀.")

rows = []
for path in tqdm(test_files, desc="Test files", ncols=100):
    df = pd.read_csv(path)
    need = {"date","store","menu","store_menu","sales"}
    if not need.issubset(df.columns):
        raise ValueError(f"{os.path.basename(path)} 필수 컬럼 누락: {sorted(need - set(df.columns))}")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["store_menu","date"])

    for sm, g in tqdm(list(df.groupby("store_menu", sort=False)), desc=os.path.basename(path), leave=False, ncols=100, unit="series"):
        vals = g["sales"].astype(float).clip(lower=0).values
        dates = pd.DatetimeIndex(g["date"].values)
        if len(vals) < BACKCAST_LEN:
            pad = BACKCAST_LEN - len(vals)
            vals = np.concatenate([np.zeros(pad, dtype=np.float32), vals])
            dates = pd.DatetimeIndex(list(pd.date_range(dates.min()-pd.Timedelta(days=pad), periods=pad, freq="D")) + list(dates))
        else:
            vals = vals[-BACKCAST_LEN:]
            dates = dates[-BACKCAST_LEN:]
        fut_dates, yhat7 = forecast_with_exog(sm, dates, vals)
        for d, y in zip(fut_dates, yhat7):
            rows.append({"test_file": os.path.basename(path), "store_menu": sm, "target_date": d.date().isoformat(), "pred": float(y)})

predictions_long = pd.DataFrame(rows).sort_values(["test_file","store_menu","target_date"]).reset_index(drop=True)
out_csv = os.path.join(RESULT_DIR, "nbeatsx_exog_forecasts_long.csv")
predictions_long.to_csv(out_csv, index=False)
print(f"저장 완료: {out_csv}")
predictions_long.head()


/tmp/ipykernel_3811013/2646805126.py:248: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
[01] train:   0%|          | 0/22 [00:00<?, ?it/s]/tmp/ipykernel_3811013/2646805126.py:278: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
[01] val:   0%|          | 0/3 [00:00<?, ?it/s]                           /tmp/ipykernel_3811013/2646805126.py:299: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16 if use_amp else None):


[01] train MAE 6715.593042 | val sMAPE 1.557230 | best 1.557230 | patience 0/8


[02] train MAE 7.643478 | val sMAPE 1.444361 | best 1.444361 | patience 0/8


[03] train MAE 6.200854 | val sMAPE 1.440879 | best 1.440879 | patience 0/8


[04] train MAE 5.665782 | val sMAPE 1.410023 | best 1.410023 | patience 0/8


...

[34] train MAE 3.007042 | val sMAPE 1.367487 | best 1.362156 | patience 7/8


[35] train MAE 2.976023 | val sMAPE 1.365234 | best 1.362156 | patience 8/8
Early stopping.


Test files:   0%|                                                            | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_3811013/2646805126.py:351: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16 if use_amp else None):
/tmp/ipykernel_3811013/2646805126.py:351: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16 if use_amp else None):
/tmp/ipykernel_3811013/2646805126.py:351: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16 if use_amp else None):
/tmp/ipykernel_3811013/2646805126.py:351: FutureWarning: `torch.cuda.amp.autoc

저장 완료: ./result/nbeatsx_exog_forecasts_long.csv


,test_file,store_menu,target_date,pred
0,TEST_00.csv,느티나무 셀프BBQ_1인 수저세트,2024-07-14,4.334282
1,TEST_00.csv,느티나무 셀프BBQ_1인 수저세트,2024-07-15,0.345072
2,TEST_00.csv,느티나무 셀프BBQ_1인 수저세트,2024-07-16,1.638857
3,TEST_00.csv,느티나무 셀프BBQ_1인 수저세트,2024-07-17,1.548752
4,TEST_00.csv,느티나무 셀프BBQ_1인 수저세트,2024-07-18,1.110967
